# Bayesian Reinforcement Learning Model

This notebook builds a trial-by-trial Rescorla-Wagner Q-learning model using PyMC to explain participant behavior.
It estimates learning rate ($alpha$) and inverse temperature ($eta$) for each subject and session independently.


In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import pytensor
import pytensor.tensor as pt
import matplotlib.pyplot as plt
import arviz as az
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load Data
file_path = '../data/PSUB2_RL_data_and_questionnaire_all_subs_4.21.26.xlsx'
df = pd.read_excel(file_path)

# Filter out trials with missing choices
df = df.dropna(subset=['ChoiceKey']).copy()

# Map stimulus IDs to integers 0, 1, 2, 3
stim_map = {'W1': 0, 'W2': 1, 'L1': 2, 'L2': 3}
inv_stim_map = {v: k for k, v in stim_map.items()}

df['LeftID_int'] = df['LeftID'].map(stim_map)
df['RightID_int'] = df['RightID'].map(stim_map)
df['ChosenID_int'] = df['ChosenID'].map(stim_map)

# Map ChoiceKey to 0 (Left) and 1 (Right)
df['Choice'] = df['ChoiceKey'] - 1

# Scale rewards for more stable sampling
# Dividing by 30 scales the max reward (+30) to 1.0, and max loss (-30) to -1.0
df['RewardScaled'] = df['TrialPoints'] / 30.0

print(f"Total trials across all subjects: {len(df)}")


In [ ]:
def run_rl_model(df_sub):
    """
    Builds and samples a PyMC RL model for a single subject and session.
    """
    df_sub = df_sub.sort_values('Trial')
    
    left_id = df_sub['LeftID_int'].values.astype(int)
    right_id = df_sub['RightID_int'].values.astype(int)
    choice = df_sub['Choice'].values.astype(int)
    reward = df_sub['RewardScaled'].values
    chosen_id = df_sub['ChosenID_int'].values.astype(int)
    
    trials = len(df_sub)
    
    def update_Q(left, right, chosen, rew, Q_t, alpha):
        Q_t_next = pt.set_subtensor(Q_t[chosen], Q_t[chosen] + alpha * (rew - Q_t[chosen]))
        return Q_t_next

    with pm.Model() as rl_model:
        # Priors
        alpha = pm.Beta('alpha', alpha=1, beta=1)
        beta = pm.HalfNormal('beta', sigma=10)
        
        Q_init = pt.zeros(4)
        
        # Trial-by-trial update
        Q_seq = pytensor.scan(
            fn=update_Q,
            sequences=[
                pt.as_tensor_variable(left_id),
                pt.as_tensor_variable(right_id),
                pt.as_tensor_variable(chosen_id),
                pt.as_tensor_variable(reward)
            ],
            outputs_info=[Q_init],
            non_sequences=[alpha],
            strict=True,
            return_updates=False
        )
        
        # Prepend initial Q values to align with trials
        Q_start = pt.concatenate([pt.shape_padleft(Q_init), Q_seq[:-1]], axis=0)
        
        # Add deterministic variable to save Q-values
        pm.Deterministic('Q_values', Q_start)
        
        # Extract Q-values for presented options
        Q_left = Q_start[pt.arange(trials), left_id]
        Q_right = Q_start[pt.arange(trials), right_id]
        
        # Probability of choosing Right (Choice == 1)
        p_right = pm.math.sigmoid(beta * (Q_right - Q_left))
        pm.Deterministic('p_right', p_right)
        
        # Likelihood
        choice_obs = pm.Bernoulli('choice_obs', p=p_right, observed=choice)
        
        # Sample
        idata = pm.sample(draws=1000, tune=1000, cores=4, progressbar=False)
        
    return idata, rl_model


In [ ]:
def plot_session(df_sub, idata, sub, ses):
    """
    Plots the Q-values, choices, and probabilities separately for W and L trials.
    """
    df_sub = df_sub.sort_values('Trial').reset_index(drop=True)
    
    # Extract posterior means
    Q_mean = idata.posterior['Q_values'].mean(dim=['chain', 'draw']).values
    p_right_mean = idata.posterior['p_right'].mean(dim=['chain', 'draw']).values
    
    alpha_mean = idata.posterior['alpha'].mean().item()
    beta_mean = idata.posterior['beta'].mean().item()
    
    # Add predicted probabilities for W1 and L1 specifically
    # For a given trial, we want P(Choice == W1) or P(Choice == L1)
    df_sub['P_Right'] = p_right_mean
    
    # Compute P(Choice == Option1) where Option1 is W1 or L1
    # If W1 is Right, P(W1) = P_Right. If W1 is Left, P(W1) = 1 - P_Right.
    def get_p_opt1(row, opt1_id):
        if row['RightID'] == opt1_id:
            return row['P_Right']
        elif row['LeftID'] == opt1_id:
            return 1.0 - row['P_Right']
        return np.nan
        
    df_sub['P_W1'] = df_sub.apply(lambda r: get_p_opt1(r, 'W1') if r['TrialType'] == 'W' else np.nan, axis=1)
    df_sub['P_L1'] = df_sub.apply(lambda r: get_p_opt1(r, 'L1') if r['TrialType'] == 'L' else np.nan, axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    fig.suptitle(f"Subject {sub} - Session {ses} | $\\alpha$={alpha_mean:.2f}, $\\beta$={beta_mean:.2f}", fontsize=14)
    
    # --- Positive Trials (W) ---
    ax_w = axes[0]
    df_w = df_sub[df_sub['TrialType'] == 'W']
    
    if not df_w.empty:
        x_w = df_w['IdxWithinType'].values
        
        # Plot choices: Y=1 if ChosenID == W1, Y=0 if ChosenID == W2
        choices_w = (df_w['ChosenID'] == 'W1').astype(int).values
        ax_w.scatter(x_w, choices_w, color='black', alpha=0.6, label='Choice (1=W1, 0=W2)')
        
        # Plot P(W1)
        ax_w.plot(x_w, df_w['P_W1'].values, color='green', label='P(Choose W1)', lw=2)
        
        ax_w.set_title("Win Domain (W1 vs W2)")
        ax_w.set_xlabel("IdxWithinType")
        ax_w.set_yticks([0, 1])
        ax_w.set_yticklabels(['W2', 'W1'])
        ax_w.set_ylim(-0.1, 1.1)
        
        # Plot Q-values on secondary axis
        ax_qw = ax_w.twinx()
        q_w1 = Q_mean[df_w.index, 0] # W1 is 0
        q_w2 = Q_mean[df_w.index, 1] # W2 is 1
        ax_qw.plot(x_w, q_w1, color='blue', linestyle='--', label='Q(W1)')
        ax_qw.plot(x_w, q_w2, color='orange', linestyle='--', label='Q(W2)')
        ax_qw.set_ylabel("Q-value (Scaled)")
        
        # Combine legends
        lines, labels = ax_w.get_legend_handles_labels()
        lines2, labels2 = ax_qw.get_legend_handles_labels()
        ax_w.legend(lines + lines2, labels + labels2, loc='center right')
        
    # --- Negative Trials (L) ---
    ax_l = axes[1]
    df_l = df_sub[df_sub['TrialType'] == 'L']
    
    if not df_l.empty:
        x_l = df_l['IdxWithinType'].values
        
        # Plot choices: Y=1 if ChosenID == L1, Y=0 if ChosenID == L2
        choices_l = (df_l['ChosenID'] == 'L1').astype(int).values
        ax_l.scatter(x_l, choices_l, color='black', alpha=0.6, label='Choice (1=L1, 0=L2)')
        
        # Plot P(L1)
        ax_l.plot(x_l, df_l['P_L1'].values, color='red', label='P(Choose L1)', lw=2)
        
        ax_l.set_title("Loss Domain (L1 vs L2)")
        ax_l.set_xlabel("IdxWithinType")
        ax_l.set_yticks([0, 1])
        ax_l.set_yticklabels(['L2', 'L1'])
        ax_l.set_ylim(-0.1, 1.1)
        
        # Plot Q-values on secondary axis
        ax_ql = ax_l.twinx()
        q_l1 = Q_mean[df_l.index, 2] # L1 is 2
        q_l2 = Q_mean[df_l.index, 3] # L2 is 3
        ax_ql.plot(x_l, q_l1, color='purple', linestyle='--', label='Q(L1)')
        ax_ql.plot(x_l, q_l2, color='brown', linestyle='--', label='Q(L2)')
        ax_ql.set_ylabel("Q-value (Scaled)")
        
        # Combine legends
        lines, labels = ax_l.get_legend_handles_labels()
        lines2, labels2 = ax_ql.get_legend_handles_labels()
        ax_l.legend(lines + lines2, labels + labels2, loc='center right')

    plt.tight_layout()
    plt.show()


In [ ]:
# Get all unique Subject and Session combinations
sub_ses_pairs = df[['Sub', 'Ses']].drop_duplicates().sort_values(['Sub', 'Ses']).values

results = {}

# For demonstration, we'll run the first two pairs. 
# Remove `[:2]` to run for all subjects and sessions.
for sub, ses in sub_ses_pairs:
    print(f"--- Running Subject {sub}, Session {ses} ---")
    df_sub = df[(df['Sub'] == sub) & (df['Ses'] == ses)].copy()
    
    if len(df_sub) < 10:
        print("Not enough trials. Skipping.")
        continue
        
    idata, model = run_rl_model(df_sub)
    results[(sub, ses)] = idata
    
    plot_session(df_sub, idata, sub, ses)
